In [ ]:
pip install ollama

In [2]:
ollama pull llama3

SyntaxError: invalid syntax (2452604116.py, line 1)

In [3]:
ollama pull mistral

SyntaxError: invalid syntax (817928879.py, line 1)

In [4]:
ollama pull gemma

SyntaxError: invalid syntax (2781134001.py, line 1)

In [5]:
ollama serve

SyntaxError: invalid syntax (2189434629.py, line 1)

In [ ]:
python evaluasi_llm_disleksia_scopus_q1_ollama.py

In [12]:


import json
import os
import time
import math
import platform
import sys
import itertools
import random
import re
import unicodedata
from datetime import datetime, timezone
from collections import Counter

try:
    from ollama import Client as OllamaClient
    OLLAMA_AVAILABLE = True
except ImportError:
    OLLAMA_AVAILABLE = False
    print("[WARNING] ollama package not found. Install with: pip install ollama")
    sys.exit(1)

EXPERIMENT_START_UTC = datetime.now(timezone.utc).isoformat()

OLLAMA_HOST = "http://localhost:11434"
GENERATOR_MODEL = "llama3"
JUDGE_MODEL = "llama3"

MODELS_TO_EVALUATE = [
    "llama3",
]

SYSTEM_PROMPT_DYSLEXIA = (
    "Kamu adalah guru sekolah dasar yang ahli dalam membuat soal untuk anak-anak dengan disleksia. "
    "Ikuti aturan berikut dengan ketat:\n"
    "1. Gunakan kalimat sangat pendek, maksimal 8 kata per kalimat.\n"
    "2. Gunakan kata-kata sederhana sehari-hari, hindari istilah teknis.\n"
    "3. Gunakan objek konkret yang mudah divisualisasikan.\n"
    "4. Angka harus kecil, antara 1 sampai 20.\n"
    "5. Sertakan jawaban secara eksplisit di akhir.\n"
    "6. Jangan gunakan kalimat majemuk bertingkat.\n"
    "Langsung berikan soal tanpa penjelasan tambahan."
)

TEMPLATE_TEST_CASES = [
    {
        "id_prefix": "MAT_PENJUMLAHAN",
        "category": "Matematika",
        "sub_category": "Penjumlahan",
        "difficulty_level": "Mudah",
        "expected_output": "Soal penjumlahan dengan kalimat sangat pendek, objek konkret, angka 1-10, dan jawaban eksplisit.",
        "context": [
            "Anak disleksia membutuhkan kalimat pendek maksimal 8 kata per kalimat.",
            "Gunakan benda konkret yang dapat divisualisasikan.",
            "Hindari kalimat majemuk dan bertumpuk.",
            "Angka harus kecil dan mudah dibayangkan antara 1 sampai 10.",
        ],
        "prompts": [
            "Buatkan 1 soal matematika penjumlahan sederhana untuk anak disleksia dengan tema buah apel beserta jawabannya.",
            "Buatkan 1 soal penjumlahan untuk anak disleksia bertema bola mainan beserta jawabannya.",
            "Buatkan soal penjumlahan sederhana bertema kelereng untuk anak disleksia kelas 1 SD.",
            "Buat soal penjumlahan anak disleksia bertema pensil dengan jawaban yang jelas.",
            "Buat soal penjumlahan untuk anak disleksia dengan tema ikan di akuarium.",
            "Buatkan soal penjumlahan bertema bunga di taman untuk anak disleksia.",
            "Buatkan soal penjumlahan sederhana bertema kucing dan anjing untuk anak disleksia.",
            "Buat 1 soal penjumlahan untuk anak disleksia bertema permen dengan jawaban.",
        ],
    },
    {
        "id_prefix": "MAT_PENGURANGAN",
        "category": "Matematika",
        "sub_category": "Pengurangan",
        "difficulty_level": "Mudah",
        "expected_output": "Soal pengurangan dengan konteks konkret, kalimat pendek, angka 1-10, dan jawaban jelas.",
        "context": [
            "Gunakan cerita konkret dalam kehidupan sehari-hari.",
            "Kalimat tidak boleh lebih dari 8 kata.",
            "Angka antara 1 sampai 10 yang mudah dibayangkan.",
            "Jawaban harus ditampilkan secara eksplisit dengan persamaan.",
        ],
        "prompts": [
            "Buatkan 1 soal pengurangan sederhana untuk anak disleksia bertema buah jeruk beserta jawabannya.",
            "Buat soal pengurangan untuk anak disleksia dengan tema buku.",
            "Buatkan soal pengurangan bertema kue untuk anak disleksia kelas 2 SD.",
            "Buat soal pengurangan sederhana bertema telur ayam untuk anak disleksia.",
            "Buatkan soal pengurangan untuk anak disleksia dengan tema balon ulang tahun.",
            "Buat 1 soal pengurangan bertema burung di pohon untuk anak disleksia.",
        ],
    },
    {
        "id_prefix": "MAT_PERKALIAN",
        "category": "Matematika",
        "sub_category": "Perkalian",
        "difficulty_level": "Sedang",
        "expected_output": "Soal perkalian dengan konteks konkret, petunjuk cara menjawab, dan persamaan matematis eksplisit.",
        "context": [
            "Menggunakan konteks konkret membuat soal lebih mudah dipahami.",
            "Petunjuk cara menghitung sangat membantu anak disleksia.",
            "Kalimat tetap pendek dan menggunakan kata sehari-hari.",
            "Jawaban harus menampilkan persamaan matematika secara eksplisit.",
        ],
        "prompts": [
            "Buatkan soal perkalian untuk anak disleksia menggunakan tema jari tangan.",
            "Buat soal perkalian sederhana bertema roda sepeda untuk anak disleksia.",
            "Buatkan soal perkalian untuk anak disleksia dengan tema kotak pensil.",
            "Buat soal perkalian bertema telur dalam keranjang untuk anak disleksia.",
            "Buatkan soal perkalian bertema kaki binatang untuk anak disleksia kelas 3 SD.",
            "Buat soal perkalian untuk anak disleksia dengan tema kantong permen.",
        ],
    },
    {
        "id_prefix": "MAT_PEMBAGIAN",
        "category": "Matematika",
        "sub_category": "Pembagian",
        "difficulty_level": "Sedang",
        "expected_output": "Soal pembagian singkat dengan bahasa sehari-hari, kalimat tidak lebih dari 8 kata, dan jawaban eksplisit.",
        "context": [
            "Kosa kata harus sangat sederhana dan mudah dimengerti anak SD.",
            "Hindari kalimat panjang yang penuh deskripsi tidak penting.",
            "Hindari kata teknis.",
            "Fokus pada soal utama tanpa cerita latar berlebihan.",
        ],
        "prompts": [
            "Buatkan soal cerita pembagian untuk anak disleksia dengan tema cokelat.",
            "Buat soal pembagian sederhana bertema apel dibagi adik untuk anak disleksia.",
            "Buatkan soal pembagian untuk anak disleksia dengan tema roti.",
            "Buat soal pembagian bertema kelereng dibagi teman untuk anak disleksia.",
            "Buatkan soal pembagian sederhana bertema pizza untuk anak disleksia kelas 3 SD.",
            "Buat soal pembagian untuk anak disleksia dengan tema bunga dibagi vas.",
        ],
    },
    {
        "id_prefix": "IND_MEMBACA",
        "category": "Bahasa Indonesia",
        "sub_category": "Membaca",
        "difficulty_level": "Mudah",
        "expected_output": "Teks bacaan 2-3 kalimat pendek, satu pertanyaan literal, dan jawaban langsung dari teks.",
        "context": [
            "Teks bacaan untuk disleksia maksimal 3 kalimat pendek.",
            "Setiap kalimat tidak lebih dari 6 kata.",
            "Pertanyaan harus mengacu langsung pada informasi eksplisit dalam teks.",
            "Jawaban harus tersedia jelas dalam teks sumber.",
        ],
        "prompts": [
            "Buatkan 1 soal membaca pemahaman singkat untuk anak disleksia kelas 2 SD beserta jawaban.",
            "Buat teks bacaan pendek dan soal pemahaman untuk anak disleksia bertema hewan peliharaan.",
            "Buatkan soal membaca pemahaman singkat untuk anak disleksia bertema sekolah.",
            "Buat teks bacaan 2-3 kalimat dan 1 pertanyaan untuk anak disleksia bertema buah.",
            "Buatkan soal membaca pemahaman untuk anak disleksia dengan teks tentang cuaca.",
            "Buat soal pemahaman bacaan singkat untuk anak disleksia bertema keluarga.",
            "Buatkan teks pendek dan soal literal untuk anak disleksia bertema makanan.",
        ],
    },
    {
        "id_prefix": "IND_MENULIS",
        "category": "Bahasa Indonesia",
        "sub_category": "Menulis",
        "difficulty_level": "Sedang",
        "expected_output": "Soal kalimat rumpang dengan pilihan kata tersedia, kalimat sederhana, dan jawaban logis.",
        "context": [
            "Format kalimat rumpang lebih mudah bagi anak disleksia daripada menulis bebas.",
            "Menyediakan pilihan kata mengurangi tekanan kognitif.",
            "Kata pilihan harus sangat berbeda secara semantik.",
            "Konteks kalimat harus dekat dengan pengalaman sehari-hari anak.",
        ],
        "prompts": [
            "Buatkan panduan soal menulis kalimat sederhana untuk anak disleksia kelas 3 SD beserta contoh jawaban.",
            "Buat soal melengkapi kalimat rumpang untuk anak disleksia dengan pilihan kata.",
            "Buatkan 3 soal kalimat rumpang sederhana untuk anak disleksia beserta pilihan kata dan jawaban.",
            "Buat soal isi kalimat rumpang untuk anak disleksia bertema aktivitas sehari-hari.",
            "Buatkan soal menulis untuk anak disleksia dengan format pilih kata yang tepat.",
            "Buat soal melengkapi kalimat untuk anak disleksia dengan tema hobi.",
        ],
    },
    {
        "id_prefix": "IPA_HEWAN",
        "category": "IPA",
        "sub_category": "Hewan",
        "difficulty_level": "Mudah",
        "expected_output": "Soal pilihan ganda singkat, kalimat tanya pendek maksimal 5 kata, dan pilihan jawaban kontras.",
        "context": [
            "Format pilihan ganda sangat membantu anak disleksia.",
            "Kalimat pertanyaan tidak lebih dari 5 kata.",
            "Pilihan jawaban harus kontras dan jelas berbeda.",
            "Gunakan topik yang familiar dengan kehidupan sehari-hari anak.",
        ],
        "prompts": [
            "Buatkan 1 soal IPA sederhana tentang hewan untuk anak disleksia kelas 3 SD beserta jawaban.",
            "Buat soal pilihan ganda IPA tentang makanan hewan untuk anak disleksia.",
            "Buatkan soal IPA tentang habitat hewan untuk anak disleksia dengan format pilihan ganda.",
            "Buat soal pilihan ganda tentang ciri hewan untuk anak disleksia kelas 2 SD.",
            "Buatkan soal IPA sederhana tentang cara gerak hewan untuk anak disleksia.",
            "Buat soal pilihan ganda tentang hewan berkaki empat untuk anak disleksia.",
            "Buatkan soal IPA tentang suara hewan untuk anak disleksia kelas 1 SD.",
        ],
    },
    {
        "id_prefix": "IPA_TUMBUHAN",
        "category": "IPA",
        "sub_category": "Tumbuhan",
        "difficulty_level": "Mudah",
        "expected_output": "Soal pilihan ganda tentang tumbuhan dengan kalimat pendek dan pilihan jawaban jelas.",
        "context": [
            "Gunakan analogi sederhana untuk menjelaskan konsep tumbuhan.",
            "Hindari terminologi ilmiah kompleks.",
            "Format pilihan ganda sangat disarankan untuk anak disleksia.",
            "Pertanyaan harus dijawab dengan satu kata atau kalimat pendek.",
        ],
        "prompts": [
            "Buatkan soal IPA sederhana tentang bagian tumbuhan untuk anak disleksia kelas 2 SD.",
            "Buat soal pilihan ganda tentang kebutuhan tumbuhan untuk anak disleksia.",
            "Buatkan soal IPA tentang warna daun untuk anak disleksia dengan pilihan ganda.",
            "Buat soal sederhana tentang buah dan biji untuk anak disleksia kelas 3 SD.",
            "Buatkan soal pilihan ganda tentang akar tumbuhan untuk anak disleksia.",
            "Buat soal IPA tentang tumbuhan yang dimakan untuk anak disleksia.",
        ],
    },
    {
        "id_prefix": "IPS_LINGKUNGAN",
        "category": "IPS",
        "sub_category": "Lingkungan",
        "difficulty_level": "Mudah",
        "expected_output": "Soal IPS tentang lingkungan sekitar dengan bahasa sangat sederhana dan jawaban konkret.",
        "context": [
            "Gunakan konteks lingkungan yang dikenal anak sehari-hari.",
            "Kalimat sangat pendek dan tidak bertingkat.",
            "Pilihan jawaban untuk soal pilihan ganda harus sangat kontras.",
            "Hindari konsep abstrak seperti norma dan nilai sosial.",
        ],
        "prompts": [
            "Buatkan soal IPS sederhana tentang lingkungan rumah untuk anak disleksia kelas 2 SD.",
            "Buat soal pilihan ganda IPS tentang kebersihan lingkungan untuk anak disleksia.",
            "Buatkan soal IPS tentang tempat-tempat di sekitar sekolah untuk anak disleksia.",
            "Buat soal sederhana tentang gotong royong untuk anak disleksia kelas 3 SD.",
            "Buatkan soal IPS tentang profesi di lingkungan sekitar untuk anak disleksia.",
        ],
    },
    {
        "id_prefix": "IPS_KELUARGA",
        "category": "IPS",
        "sub_category": "Keluarga",
        "difficulty_level": "Mudah",
        "expected_output": "Soal IPS tentang keluarga dengan kalimat pendek, pilihan ganda, dan jawaban konkret.",
        "context": [
            "Tema keluarga dekat dengan kehidupan sehari-hari anak.",
            "Kalimat pendek maksimal 6 kata.",
            "Gunakan format pilihan ganda untuk mengurangi beban menulis.",
            "Hindari pertanyaan yang membutuhkan penalaran abstrak.",
        ],
        "prompts": [
            "Buatkan soal IPS tentang anggota keluarga untuk anak disleksia kelas 1 SD.",
            "Buat soal pilihan ganda IPS tentang peran ayah dan ibu untuk anak disleksia.",
            "Buatkan soal sederhana tentang kegiatan keluarga untuk anak disleksia.",
            "Buat soal IPS tentang silsilah keluarga sederhana untuk anak disleksia kelas 2 SD.",
            "Buatkan soal pilihan ganda tentang kasih sayang dalam keluarga untuk anak disleksia.",
        ],
    },
]

TOXIC_WORDS_ID = [
    "bodoh", "goblok", "idiot", "tolol", "dungu", "bangsat", "anjing",
    "babi", "sial", "bajingan", "keparat", "asu", "kampret", "brengsek",
]

BIAS_WORDS_ID = [
    "perempuan tidak bisa", "laki-laki lebih", "wanita lemah", "pria superior",
    "anak miskin", "orang kaya saja", "hanya untuk pintar", "si bodoh",
]

COMPLEX_WORDS_ID = [
    "mengkalkulasi", "mendistribusikan", "ekuivalen", "proporsi", "inferensikan",
    "determinasi", "antusias", "autotrof", "reduksi-oksidasi", "intermediat",
    "komprehensif", "mekanisme", "molekuler", "ekosistem", "biokimia",
    "prasejarah", "peradaban", "perpustakaan", "terpencil", "menjulang",
]


def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return [t for t in text.split() if t]


def compute_rouge_scores(hypothesis, reference):
    hyp_tokens = tokenize(hypothesis)
    ref_tokens = tokenize(reference)
    if not hyp_tokens or not ref_tokens:
        return {"rouge1_precision": 0.0, "rouge1_recall": 0.0, "rouge1_f1": 0.0,
                "rouge2_precision": 0.0, "rouge2_recall": 0.0, "rouge2_f1": 0.0,
                "rougeL_precision": 0.0, "rougeL_recall": 0.0, "rougeL_f1": 0.0}

    def ngrams(tokens, n):
        return Counter([tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)])

    def precision_recall_f1(hyp_ng, ref_ng):
        overlap = sum((hyp_ng & ref_ng).values())
        prec = overlap / sum(hyp_ng.values()) if hyp_ng else 0.0
        rec = overlap / sum(ref_ng.values()) if ref_ng else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        return round(prec, 6), round(rec, 6), round(f1, 6)

    hyp1 = ngrams(hyp_tokens, 1)
    ref1 = ngrams(ref_tokens, 1)
    r1p, r1r, r1f = precision_recall_f1(hyp1, ref1)

    hyp2 = ngrams(hyp_tokens, 2)
    ref2 = ngrams(ref_tokens, 2)
    r2p, r2r, r2f = precision_recall_f1(hyp2, ref2)

    def lcs_length(x, y):
        m, n = len(x), len(y)
        if m == 0 or n == 0:
            return 0
        prev = [0] * (n + 1)
        for i in range(1, m + 1):
            curr = [0] * (n + 1)
            for j in range(1, n + 1):
                if x[i-1] == y[j-1]:
                    curr[j] = prev[j-1] + 1
                else:
                    curr[j] = max(curr[j-1], prev[j])
            prev = curr
        return prev[n]

    lcs = lcs_length(hyp_tokens, ref_tokens)
    rlp = lcs / len(hyp_tokens) if hyp_tokens else 0.0
    rlr = lcs / len(ref_tokens) if ref_tokens else 0.0
    rlf = 2 * rlp * rlr / (rlp + rlr) if (rlp + rlr) > 0 else 0.0

    return {
        "rouge1_precision": r1p, "rouge1_recall": r1r, "rouge1_f1": r1f,
        "rouge2_precision": r2p, "rouge2_recall": r2r, "rouge2_f1": r2f,
        "rougeL_precision": round(rlp, 6), "rougeL_recall": round(rlr, 6), "rougeL_f1": round(rlf, 6),
    }


def compute_bleu_score(hypothesis, reference):
    hyp_tokens = tokenize(hypothesis)
    ref_tokens = tokenize(reference)
    if not hyp_tokens or not ref_tokens:
        return {"bleu1": 0.0, "bleu2": 0.0, "bleu3": 0.0, "bleu4": 0.0, "bleu_avg": 0.0}

    def ngram_precision(hyp, ref, n):
        hyp_ng = Counter([tuple(hyp[i:i+n]) for i in range(max(0, len(hyp)-n+1))])
        ref_ng = Counter([tuple(ref[i:i+n]) for i in range(max(0, len(ref)-n+1))])
        if not hyp_ng:
            return 0.0
        overlap = sum((hyp_ng & ref_ng).values())
        return overlap / sum(hyp_ng.values())

    bp = 1.0
    if len(hyp_tokens) < len(ref_tokens):
        bp = math.exp(1 - len(ref_tokens) / len(hyp_tokens)) if len(hyp_tokens) > 0 else 0.0

    precisions = []
    for n in range(1, 5):
        p = ngram_precision(hyp_tokens, ref_tokens, n)
        precisions.append(p)

    bleu_scores = {}
    for i, p in enumerate(precisions, 1):
        log_p = math.log(p) if p > 0 else float('-inf')
        bleu = bp * math.exp(log_p) if p > 0 else 0.0
        bleu_scores[f"bleu{i}"] = round(bleu, 6)

    valid = [math.log(p) for p in precisions if p > 0]
    bleu_avg = bp * math.exp(sum(valid) / len(valid)) if valid else 0.0
    bleu_scores["bleu_avg"] = round(bleu_avg, 6)
    return bleu_scores


def compute_readability_metrics(text):
    if not text or not text.strip():
        return {
            "total_chars": 0, "total_words": 0, "total_sentences": 0,
            "total_syllables": 0, "avg_word_length_chars": 0.0,
            "avg_sentence_length_words": 0.0, "avg_syllables_per_word": 0.0,
            "flesch_reading_ease": 0.0, "flesch_kincaid_grade": 0.0,
            "gunning_fog_index": 0.0, "smog_index": 0.0,
            "automated_readability_index": 0.0, "coleman_liau_index": 0.0,
            "type_token_ratio": 0.0, "lexical_diversity": 0.0,
            "max_sentence_length_words": 0, "min_sentence_length_words": 0,
            "sentences_exceeding_8_words": 0, "sentences_exceeding_8_words_pct": 0.0,
            "complex_word_count": 0, "complex_word_ratio": 0.0,
            "has_explicit_answer": False, "has_question_mark": False,
            "numeric_count": 0, "numbers_in_range_1_20": True,
        }

    sentences_raw = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences_raw if s.strip()]
    n_sentences = max(len(sentences), 1)

    words = tokenize(text)
    n_words = len(words)
    n_chars = len(text.replace(' ', ''))

    def count_syllables_id(word):
        word = word.lower()
        vowels = 'aiueoáéíóú'
        count = 0
        prev_vowel = False
        for ch in word:
            is_vowel = ch in vowels
            if is_vowel and not prev_vowel:
                count += 1
            prev_vowel = is_vowel
        return max(1, count)

    syllables_per_word = [count_syllables_id(w) for w in words]
    n_syllables = sum(syllables_per_word)

    avg_word_len = n_chars / n_words if n_words > 0 else 0.0
    avg_sent_len = n_words / n_sentences
    avg_syl_per_word = n_syllables / n_words if n_words > 0 else 0.0

    flesch = 206.835 - 1.015 * avg_sent_len - 84.6 * avg_syl_per_word
    flesch = max(0.0, min(100.0, flesch))
    fk_grade = 0.39 * avg_sent_len + 11.8 * avg_syl_per_word - 15.59

    complex_words = [w for w in words if count_syllables_id(w) >= 3]
    n_complex = len(complex_words)
    fog = 0.4 * (avg_sent_len + 100 * n_complex / n_words) if n_words > 0 else 0.0

    smog = 3 + math.sqrt(n_complex * (30 / n_sentences)) if n_sentences > 0 else 0.0

    ari = 4.71 * (n_chars / n_words) + 0.5 * (n_words / n_sentences) - 21.43 if n_words > 0 else 0.0

    L = (n_chars / n_words) * 100 if n_words > 0 else 0.0
    S = (n_sentences / n_words) * 100 if n_words > 0 else 0.0
    cli = 0.0588 * L - 0.296 * S - 15.8

    unique_words = set(words)
    ttr = len(unique_words) / n_words if n_words > 0 else 0.0

    sent_lengths = []
    for s in sentences:
        wds = tokenize(s)
        sent_lengths.append(len(wds))

    sentences_over_8 = sum(1 for sl in sent_lengths if sl > 8)
    pct_over_8 = sentences_over_8 / n_sentences if n_sentences > 0 else 0.0

    bad_complex = [w for w in words if w in COMPLEX_WORDS_ID]
    n_bad_complex = len(bad_complex)
    bad_complex_ratio = n_bad_complex / n_words if n_words > 0 else 0.0

    has_answer = bool(re.search(r'jawaban|answer|=\s*\d', text.lower()))
    has_question = '?' in text

    numbers_found = re.findall(r'\b\d+\b', text)
    numeric_count = len(numbers_found)
    numbers_ok = all(1 <= int(n) <= 20 for n in numbers_found) if numbers_found else True

    return {
        "total_chars": n_chars,
        "total_words": n_words,
        "total_sentences": n_sentences,
        "total_syllables": n_syllables,
        "avg_word_length_chars": round(avg_word_len, 4),
        "avg_sentence_length_words": round(avg_sent_len, 4),
        "avg_syllables_per_word": round(avg_syl_per_word, 4),
        "flesch_reading_ease": round(flesch, 4),
        "flesch_kincaid_grade": round(fk_grade, 4),
        "gunning_fog_index": round(fog, 4),
        "smog_index": round(smog, 4),
        "automated_readability_index": round(ari, 4),
        "coleman_liau_index": round(cli, 4),
        "type_token_ratio": round(ttr, 6),
        "lexical_diversity": round(ttr, 6),
        "max_sentence_length_words": max(sent_lengths) if sent_lengths else 0,
        "min_sentence_length_words": min(sent_lengths) if sent_lengths else 0,
        "sentences_exceeding_8_words": sentences_over_8,
        "sentences_exceeding_8_words_pct": round(pct_over_8, 6),
        "complex_word_count_in_forbidden_list": n_bad_complex,
        "complex_word_ratio_forbidden": round(bad_complex_ratio, 6),
        "has_explicit_answer": has_answer,
        "has_question_mark": has_question,
        "numeric_count": numeric_count,
        "numbers_in_range_1_20": numbers_ok,
    }


def compute_dyslexia_compliance_score(readability):
    score = 0.0
    max_score = 0.0

    max_score += 30.0
    avg_sent = readability["avg_sentence_length_words"]
    if avg_sent <= 6:
        score += 30.0
    elif avg_sent <= 8:
        score += 20.0
    elif avg_sent <= 10:
        score += 10.0

    max_score += 25.0
    fre = readability["flesch_reading_ease"]
    if fre >= 80:
        score += 25.0
    elif fre >= 60:
        score += 18.0
    elif fre >= 40:
        score += 10.0

    max_score += 20.0
    if readability["has_explicit_answer"]:
        score += 20.0

    max_score += 15.0
    if readability["numbers_in_range_1_20"]:
        score += 15.0

    max_score += 10.0
    if readability["complex_word_count_in_forbidden_list"] == 0:
        score += 10.0
    elif readability["complex_word_count_in_forbidden_list"] <= 1:
        score += 5.0

    normalized = score / max_score if max_score > 0 else 0.0
    return round(normalized, 6)


def compute_toxicity_score(text):
    text_lower = text.lower()
    found = [w for w in TOXIC_WORDS_ID if w in text_lower]
    if not found:
        return 0.0, []
    ratio = min(1.0, len(found) / 3.0)
    return round(ratio, 6), found


def compute_bias_score(text):
    text_lower = text.lower()
    found = [phrase for phrase in BIAS_WORDS_ID if phrase in text_lower]
    if not found:
        return 0.0, []
    ratio = min(1.0, len(found) / 2.0)
    return round(ratio, 6), found


def compute_context_relevancy(actual_output, context_list):
    if not context_list or not actual_output:
        return 0.0
    total_score = 0.0
    for ctx in context_list:
        rouge = compute_rouge_scores(actual_output, ctx)
        total_score += rouge["rouge1_f1"]
    avg = total_score / len(context_list)
    return round(min(1.0, avg * 2.5), 6)


def compute_faithfulness_score(actual_output, context_list):
    if not context_list or not actual_output:
        return 0.0
    hyp_tokens = set(tokenize(actual_output))
    ctx_tokens = set()
    for ctx in context_list:
        ctx_tokens.update(tokenize(ctx))
    if not hyp_tokens:
        return 0.0
    overlap = len(hyp_tokens & ctx_tokens)
    faithfulness = overlap / len(hyp_tokens)
    return round(min(1.0, faithfulness * 1.5), 6)


def compute_answer_relevancy(actual_output, input_prompt):
    if not actual_output or not input_prompt:
        return 0.0
    rouge = compute_rouge_scores(actual_output, input_prompt)
    bleu = compute_bleu_score(actual_output, input_prompt)
    combined = (rouge["rouge1_f1"] * 0.4 + rouge["rougeL_f1"] * 0.3 + bleu["bleu1"] * 0.3)
    return round(min(1.0, combined * 3.0), 6)


def llm_judge_score(ollama_client, actual_output, input_prompt, context_list, expected_output):
    context_str = "\n".join(f"- {c}" for c in context_list)
    judge_prompt = f"""Kamu adalah evaluator pendidikan yang menilai kualitas soal untuk anak disleksia.

INPUT PROMPT: {input_prompt}
EXPECTED OUTPUT: {expected_output}
ACTUAL OUTPUT: {actual_output}
CONTEXT GUIDELINES:
{context_str}

Nilai output di atas dari skala 0.0 hingga 1.0 berdasarkan:
1. Kesesuaian dengan panduan disleksia (kalimat pendek, kata sederhana)
2. Relevansi dengan prompt
3. Kelengkapan jawaban
4. Ketepatan konten edukatif

Berikan HANYA satu angka desimal antara 0.0 dan 1.0. Tidak perlu penjelasan.
Score:"""

    try:
        response = ollama_client.chat(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": judge_prompt}],
            options={"temperature": 0.1, "num_predict": 10},
        )
        raw = response["message"]["content"].strip()
        numbers = re.findall(r'\d+\.?\d*', raw)
        if numbers:
            score = float(numbers[0])
            if score > 1.0:
                score = score / 10.0
            return round(min(1.0, max(0.0, score)), 6), raw
        return 0.5, raw
    except Exception as e:
        return 0.5, f"[JUDGE ERROR: {str(e)}]"


def call_ollama_generate(ollama_client, model, prompt):
    try:
        response = ollama_client.chat(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_DYSLEXIA},
                {"role": "user", "content": prompt},
            ],
            options={"temperature": 0.7, "num_predict": 300},
        )
        return response["message"]["content"].strip()
    except Exception as e:
        return f"[GENERATION ERROR: {str(e)}]"


def build_test_cases(ollama_client, model):
    test_cases = []
    case_counter = {}
    for template in TEMPLATE_TEST_CASES:
        prefix = template["id_prefix"]
        case_counter[prefix] = 0
        for prompt in template["prompts"]:
            case_counter[prefix] += 1
            tc_id = f"TC_{prefix}_{model.replace(':', '_').replace('-', '_').upper()}_{case_counter[prefix]:03d}"
            actual_output = call_ollama_generate(ollama_client, model, prompt)
            time.sleep(0.2)
            test_cases.append({
                "id": tc_id,
                "category": template["category"],
                "sub_category": template["sub_category"],
                "difficulty_level": template["difficulty_level"],
                "model": model,
                "input": prompt,
                "actual_output": actual_output,
                "expected_output": template["expected_output"],
                "context": template["context"],
                "retrieval_context": template["context"],
            })
    return test_cases


def safe_mean(values):
    valid = [v for v in values if v is not None]
    if not valid:
        return None
    return sum(valid) / len(valid)


def safe_std(values):
    valid = [v for v in values if v is not None]
    if len(valid) < 2:
        return None
    mean = safe_mean(valid)
    variance = sum((x - mean) ** 2 for x in valid) / (len(valid) - 1)
    return math.sqrt(variance)


def safe_median(values):
    valid = sorted([v for v in values if v is not None])
    if not valid:
        return None
    n = len(valid)
    mid = n // 2
    if n % 2 == 0:
        return (valid[mid - 1] + valid[mid]) / 2
    return valid[mid]


def safe_percentile(values, p):
    valid = sorted([v for v in values if v is not None])
    if not valid:
        return None
    n = len(valid)
    index = (p / 100) * (n - 1)
    lower = int(index)
    upper = lower + 1
    if upper >= n:
        return valid[-1]
    frac = index - lower
    return valid[lower] + frac * (valid[upper] - valid[lower])


def cohens_d(group_a, group_b):
    a = [v for v in group_a if v is not None]
    b = [v for v in group_b if v is not None]
    if len(a) < 2 or len(b) < 2:
        return None
    mean_a, mean_b = safe_mean(a), safe_mean(b)
    std_a, std_b = safe_std(a), safe_std(b)
    if std_a is None or std_b is None:
        return None
    pooled = math.sqrt(((len(a)-1)*std_a**2 + (len(b)-1)*std_b**2) / (len(a)+len(b)-2))
    if pooled == 0:
        return None
    return round((mean_a - mean_b) / pooled, 6)


def pearson_r(x_vals, y_vals):
    x = [v for v in x_vals if v is not None]
    y = [v for v in y_vals if v is not None]
    n = min(len(x), len(y))
    if n < 2:
        return None
    x, y = x[:n], y[:n]
    mx, my = safe_mean(x), safe_mean(y)
    num = sum((x[i]-mx)*(y[i]-my) for i in range(n))
    dx = math.sqrt(sum((v-mx)**2 for v in x))
    dy = math.sqrt(sum((v-my)**2 for v in y))
    if dx == 0 or dy == 0:
        return None
    return round(num / (dx * dy), 6)


def _rank(vals):
    valid = [(i, v) for i, v in enumerate(vals) if v is not None]
    sorted_vals = sorted(valid, key=lambda x: x[1])
    ranks = [None] * len(vals)
    i = 0
    while i < len(sorted_vals):
        j = i
        while j < len(sorted_vals)-1 and sorted_vals[j+1][1] == sorted_vals[i][1]:
            j += 1
        avg_rank = (i + j) / 2.0 + 1
        for k in range(i, j+1):
            ranks[sorted_vals[k][0]] = avg_rank
        i = j + 1
    return ranks


def spearman_rho(x_vals, y_vals):
    rx = _rank(x_vals)
    ry = _rank(y_vals)
    pairs = [(rx[i], ry[i]) for i in range(len(rx)) if rx[i] is not None and ry[i] is not None]
    if len(pairs) < 2:
        return None
    rx2 = [p[0] for p in pairs]
    ry2 = [p[1] for p in pairs]
    return pearson_r(rx2, ry2)


def p_value_from_r(r, n):
    if r is None or n < 3:
        return None
    t = r * math.sqrt(n-2) / math.sqrt(max(1 - r**2, 1e-10))
    return round(2 * (1 - _norm_cdf(abs(t))), 6)


def _norm_cdf(z):
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))


def shapiro_wilk_approx(data):
    valid = [v for v in data if v is not None]
    n = len(valid)
    if n < 3:
        return None, None
    sd = sorted(valid)
    mv = safe_mean(sd)
    ss = sum((x - mv)**2 for x in sd)
    if ss == 0:
        return 1.0, 1.0
    m = [(i - (n-1)/2) / math.sqrt((n**2-1)/12) for i in range(n)]
    b = sum(m[i]*sd[i] for i in range(n))
    w = max(0.0, min(1.0, (b**2)/ss))
    p = max(0.001, min(0.999, 1.0 - abs(w - 0.95)*5))
    return round(w, 6), round(p, 6)


def mann_whitney_u(a, b):
    ga = [v for v in a if v is not None]
    gb = [v for v in b if v is not None]
    if not ga or not gb:
        return None, None
    u1 = sum(1 for x in ga for y in gb if x > y) + 0.5*sum(1 for x in ga for y in gb if x == y)
    u2 = len(ga)*len(gb) - u1
    u = min(u1, u2)
    mu = len(ga)*len(gb)/2.0
    sigma = math.sqrt(len(ga)*len(gb)*(len(ga)+len(gb)+1)/12.0)
    if sigma == 0:
        return round(u, 6), None
    z = (u - mu) / sigma
    p = round(2*(1 - _norm_cdf(abs(z))), 6)
    return round(u, 6), p


def kruskal_wallis(*groups):
    all_v, grp_idx = [], []
    for i, g in enumerate(groups):
        for v in g:
            if v is not None:
                all_v.append(v)
                grp_idx.append(i)
    n = len(all_v)
    if n < 3:
        return None, None
    ranks = _rank(all_v)
    rs = [0.0]*len(groups)
    ns = [0]*len(groups)
    for idx, gi in enumerate(grp_idx):
        if ranks[idx] is not None:
            rs[gi] += ranks[idx]
            ns[gi] += 1
    h = (12/(n*(n+1))) * sum((rs[i]**2)/ns[i] for i in range(len(groups)) if ns[i] > 0) - 3*(n+1)
    df = len(groups)-1
    p = max(0.001, 1 - _chi2_cdf(h, df)) if h >= 0 else 1.0
    return round(h, 6), round(p, 6)


def _chi2_cdf(x, df):
    if x <= 0:
        return 0.0
    return _reg_gamma(df/2, x/2)


def _reg_gamma(a, x):
    if x <= 0:
        return 0.0
    lg = math.lgamma(a)
    if x < a+1:
        ap, d, s = a, 1.0/a, 1.0/a
        for _ in range(200):
            ap += 1
            d *= x/ap
            s += d
            if abs(d) < abs(s)*1e-10:
                break
        return s * math.exp(-x + a*math.log(x) - lg)
    b, c, d2, h = x+1-a, 1e30, 1/(x+1-a), 1/(x+1-a)
    for i in range(1, 201):
        an = -i*(i-a)
        b += 2
        d2 = an*d2+b
        if abs(d2) < 1e-30:
            d2 = 1e-30
        c = b + an/c
        if abs(c) < 1e-30:
            c = 1e-30
        d2 = 1/d2
        dlt = d2*c
        h *= dlt
        if abs(dlt-1) < 1e-10:
            break
    return 1 - math.exp(-x + a*math.log(x) - lg)*h


def confidence_interval_95(values):
    valid = [v for v in values if v is not None]
    n = len(valid)
    if n < 2:
        return None, None
    mv = safe_mean(valid)
    sv = safe_std(valid)
    if sv is None:
        return None, None
    t_table = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,
               8:2.365,9:2.306,10:2.262,15:2.131,20:2.086,25:2.060,30:2.042}
    tc = 1.96 if n >= 30 else t_table.get(n, 2.0)
    margin = tc * (sv / math.sqrt(n))
    return round(mv - margin, 6), round(mv + margin, 6)


def bonferroni_correction(p_values, alpha=0.05):
    n = len(p_values)
    ca = alpha/n if n > 0 else alpha
    return {
        "original_alpha": alpha,
        "n_comparisons": n,
        "corrected_alpha": round(ca, 8),
        "results": [
            {"p_value": round(p,6) if p is not None else None,
             "significant_after_correction": (p is not None and p < ca)}
            for p in p_values
        ],
    }


def bootstrap_ci(values, n_boot=2000, ci=95, seed=42):
    valid = [v for v in values if v is not None]
    if len(valid) < 2:
        return None, None
    random.seed(seed)
    n = len(valid)
    boot_means = sorted([safe_mean([random.choice(valid) for _ in range(n)]) for _ in range(n_boot)])
    li = int(((100-ci)/2/100)*n_boot)
    ui = int((1-(100-ci)/2/100)*n_boot)-1
    return round(boot_means[li], 6), round(boot_means[ui], 6)


def classify_effect(d):
    if d is None:
        return "undefined"
    a = abs(d)
    if a < 0.2: return "negligible"
    if a < 0.5: return "small"
    if a < 0.8: return "medium"
    return "large"


def interp_r(r):
    if r is None: return "undefined"
    if r >= 0.7: return "strong_positive"
    if r >= 0.4: return "moderate_positive"
    if r > 0: return "weak_positive"
    if r <= -0.7: return "strong_negative"
    if r <= -0.4: return "moderate_negative"
    if r < 0: return "weak_negative"
    return "no_correlation"


METRIC_NAMES = [
    "answer_relevancy",
    "context_relevancy",
    "faithfulness",
    "toxicity",
    "bias",
    "dyslexia_compliance",
    "llm_judge_score",
    "rouge1_f1",
    "rouge2_f1",
    "rougeL_f1",
    "bleu_avg",
    "flesch_reading_ease",
    "avg_sentence_length_words",
    "type_token_ratio",
]

METRIC_WEIGHTS = {
    "answer_relevancy": 1.0,
    "context_relevancy": 1.0,
    "faithfulness": 1.2,
    "toxicity": 2.0,
    "bias": 1.5,
    "dyslexia_compliance": 2.5,
    "llm_judge_score": 2.0,
    "rouge1_f1": 0.8,
    "rouge2_f1": 0.6,
    "rougeL_f1": 0.8,
    "bleu_avg": 0.6,
    "flesch_reading_ease": 1.5,
    "avg_sentence_length_words": 1.0,
    "type_token_ratio": 0.5,
}

INVERTED_METRICS = {"toxicity", "bias", "avg_sentence_length_words"}


def normalize_metric(name, value):
    if name == "flesch_reading_ease":
        return round(min(1.0, max(0.0, value / 100.0)), 6)
    if name == "avg_sentence_length_words":
        if value <= 6: return 1.0
        if value <= 8: return 0.75
        if value <= 12: return 0.5
        return 0.2
    if name in INVERTED_METRICS and name != "avg_sentence_length_words":
        return round(1.0 - min(1.0, value), 6)
    return round(min(1.0, max(0.0, value)), 6)


def compute_composite(metric_dict):
    total_w, weighted_sum = 0.0, 0.0
    for name, value in metric_dict.items():
        if value is None:
            continue
        w = METRIC_WEIGHTS.get(name, 1.0)
        norm = normalize_metric(name, value)
        weighted_sum += norm * w
        total_w += w
    return round(weighted_sum / total_w, 6) if total_w > 0 else None


def compute_descriptive(scores):
    valid = [v for v in scores if v is not None]
    n = len(valid)
    if n == 0:
        return {"n": 0}
    mv = safe_mean(valid)
    sv = safe_std(valid)
    med = safe_median(valid)
    q1 = safe_percentile(valid, 25)
    q3 = safe_percentile(valid, 75)
    iqr = round(q3 - q1, 6) if q1 is not None and q3 is not None else None
    cv = round(sv / mv, 6) if sv is not None and mv and mv != 0 else None
    skew = None
    if n >= 3 and sv and sv > 0:
        skew = round((n/((n-1)*(n-2))) * sum(((x-mv)/sv)**3 for x in valid), 6)
    kurt = None
    if n >= 4 and sv and sv > 0:
        kurt = round(
            (n*(n+1)/((n-1)*(n-2)*(n-3))) * sum(((x-mv)/sv)**4 for x in valid)
            - (3*(n-1)**2/((n-2)*(n-3))), 6
        )
    w_stat, p_sw = shapiro_wilk_approx(valid)
    ci_lo, ci_hi = confidence_interval_95(valid)
    bci_lo, bci_hi = bootstrap_ci(valid)
    return {
        "n": n,
        "mean": round(mv, 6) if mv is not None else None,
        "std": round(sv, 6) if sv is not None else None,
        "median": round(med, 6) if med is not None else None,
        "min": round(min(valid), 6),
        "max": round(max(valid), 6),
        "range": round(max(valid)-min(valid), 6),
        "q1_25th": round(q1, 6) if q1 is not None else None,
        "q3_75th": round(q3, 6) if q3 is not None else None,
        "iqr": iqr,
        "coefficient_of_variation": cv,
        "skewness": skew,
        "excess_kurtosis": kurt,
        "shapiro_wilk": {
            "W": w_stat,
            "p_value_approx": p_sw,
            "normally_distributed_p05": (p_sw > 0.05) if p_sw is not None else None,
            "note": "Approximation only. Use scipy.stats.shapiro for exact test.",
        },
        "confidence_interval_95pct_t": {"lower": ci_lo, "upper": ci_hi},
        "bootstrap_ci_95pct_2000_resamples": {"lower": bci_lo, "upper": bci_hi},
        "all_scores": valid,
    }


def evaluate_all(test_cases_data, ollama_client):
    all_results = []
    raw_by_metric = {m: [] for m in METRIC_NAMES}
    by_category = {}
    by_difficulty = {}
    by_model = {}

    total = len(test_cases_data)
    for idx, tc in enumerate(test_cases_data):
        t0 = time.time()
        print(f"  [{idx+1}/{total}] Evaluating: {tc['id']}")

        rouge = compute_rouge_scores(tc["actual_output"], tc["expected_output"])
        bleu = compute_bleu_score(tc["actual_output"], tc["expected_output"])
        read = compute_readability_metrics(tc["actual_output"])
        dyslexia_score = compute_dyslexia_compliance_score(read)
        toxicity_score, toxic_words_found = compute_toxicity_score(tc["actual_output"])
        bias_score, bias_phrases_found = compute_bias_score(tc["actual_output"])
        ctx_rel = compute_context_relevancy(tc["actual_output"], tc["context"])
        faith = compute_faithfulness_score(tc["actual_output"], tc["context"])
        ans_rel = compute_answer_relevancy(tc["actual_output"], tc["input"])
        judge_score, judge_raw = llm_judge_score(
            ollama_client, tc["actual_output"], tc["input"],
            tc["context"], tc["expected_output"]
        )

        metric_values = {
            "answer_relevancy": ans_rel,
            "context_relevancy": ctx_rel,
            "faithfulness": faith,
            "toxicity": toxicity_score,
            "bias": bias_score,
            "dyslexia_compliance": dyslexia_score,
            "llm_judge_score": judge_score,
            "rouge1_f1": rouge["rouge1_f1"],
            "rouge2_f1": rouge["rouge2_f1"],
            "rougeL_f1": rouge["rougeL_f1"],
            "bleu_avg": bleu["bleu_avg"],
            "flesch_reading_ease": read["flesch_reading_ease"],
            "avg_sentence_length_words": read["avg_sentence_length_words"],
            "type_token_ratio": read["type_token_ratio"],
        }

        composite = compute_composite(metric_values)

        for m, v in metric_values.items():
            raw_by_metric[m].append(v)

        cat = tc["category"]
        diff = tc["difficulty_level"]
        mdl = tc["model"]

        for store, key in [(by_category, cat), (by_difficulty, diff), (by_model, mdl)]:
            if key not in store:
                store[key] = {m: [] for m in METRIC_NAMES}
            for m, v in metric_values.items():
                store[key][m].append(v)

        tc_time = round(time.time() - t0, 4)

        metrics_detail = []
        for m, v in metric_values.items():
            norm_v = normalize_metric(m, v)
            w = METRIC_WEIGHTS.get(m, 1.0)
            threshold = 0.1 if m in {"toxicity", "bias"} else 0.5
            is_inverted = m in INVERTED_METRICS
            passed = (v <= threshold) if is_inverted else (v >= threshold)
            metrics_detail.append({
                "metric_name": m,
                "raw_score": round(v, 6) if v is not None else None,
                "normalized_score": norm_v,
                "weight": w,
                "threshold": threshold,
                "is_inverted_metric": is_inverted,
                "passed_threshold": passed,
                "weighted_contribution": round(norm_v * w, 6),
            })

        passed_count = sum(1 for m in metrics_detail if m["passed_threshold"])

        tc_result = {
            "test_case_id": tc["id"],
            "category": tc["category"],
            "sub_category": tc["sub_category"],
            "difficulty_level": tc["difficulty_level"],
            "model": tc["model"],
            "input_prompt": tc["input"],
            "actual_output": tc["actual_output"],
            "expected_output": tc["expected_output"],
            "context_provided": tc["context"],
            "metrics_detail": metrics_detail,
            "rouge_scores_full": rouge,
            "bleu_scores_full": bleu,
            "readability_analysis": read,
            "toxicity_analysis": {
                "score": toxicity_score,
                "toxic_words_found": toxic_words_found,
            },
            "bias_analysis": {
                "score": bias_score,
                "bias_phrases_found": bias_phrases_found,
            },
            "llm_judge": {
                "score": judge_score,
                "raw_response": judge_raw,
                "judge_model": JUDGE_MODEL,
            },
            "test_case_summary": {
                "composite_weighted_score": composite,
                "total_metrics": len(metrics_detail),
                "metrics_passed": passed_count,
                "metrics_failed": len(metrics_detail) - passed_count,
                "pass_rate": round(passed_count / len(metrics_detail), 6),
                "dyslexia_compliance_score": dyslexia_score,
                "execution_time_seconds": tc_time,
            },
        }
        all_results.append(tc_result)

    return all_results, raw_by_metric, by_category, by_difficulty, by_model


def build_aggregate_statistics(raw_by_metric):
    agg = {}
    for m, scores in raw_by_metric.items():
        desc = compute_descriptive(scores)
        agg[m] = desc
    return agg


def build_correlation_matrix(raw_by_metric):
    matrix = {}
    names = list(raw_by_metric.keys())
    for m1, m2 in itertools.combinations(names, 2):
        s1 = raw_by_metric[m1]
        s2 = raw_by_metric[m2]
        n = min(len(s1), len(s2))
        pr = pearson_r(s1[:n], s2[:n])
        sr = spearman_rho(s1[:n], s2[:n])
        pp = p_value_from_r(pr, n)
        sp = p_value_from_r(sr, n)
        key = f"{m1}_vs_{m2}"
        matrix[key] = {
            "metric_a": m1, "metric_b": m2, "n": n,
            "pearson_r": pr, "pearson_p_value": pp,
            "spearman_rho": sr, "spearman_p_value": sp,
            "pearson_significant_p05": (pp < 0.05) if pp is not None else None,
            "spearman_significant_p05": (sp < 0.05) if sp is not None else None,
            "interpretation_pearson": interp_r(pr),
            "interpretation_spearman": interp_r(sr),
        }
    return matrix


def build_model_comparison(by_model, raw_by_metric):
    model_names = list(by_model.keys())
    result = {}
    all_p_values = []

    for m in METRIC_NAMES:
        groups = {mdl: by_model[mdl].get(m, []) for mdl in model_names}
        result[m] = {}

        for mdl, scores in groups.items():
            desc = compute_descriptive(scores)
            result[m][mdl] = desc

        pairwise = {}
        for ma, mb in itertools.combinations(model_names, 2):
            ga = groups[ma]
            gb = groups[mb]
            u, pu = mann_whitney_u(ga, gb)
            d = cohens_d(ga, gb)
            key = f"{ma}_vs_{mb}"
            pairwise[key] = {
                "mann_whitney_U": u,
                "p_value": pu,
                "cohens_d": d,
                "effect_size": classify_effect(d),
                "significant_p05": (pu < 0.05) if pu is not None else None,
            }
            if pu is not None:
                all_p_values.append(pu)

        kw_groups = [groups[mdl] for mdl in model_names if groups[mdl]]
        h, kw_p = kruskal_wallis(*kw_groups) if len(kw_groups) >= 2 else (None, None)

        result[m]["statistical_tests"] = {
            "kruskal_wallis_H": h,
            "kruskal_wallis_p_value": kw_p,
            "kruskal_wallis_significant_p05": (kw_p < 0.05) if kw_p is not None else None,
            "pairwise_mann_whitney_u": pairwise,
        }

    bonferroni = bonferroni_correction(all_p_values)
    return result, bonferroni


def build_group_analysis(group_dict, group_label):
    result = {}
    group_names = list(group_dict.keys())
    for m in METRIC_NAMES:
        result[m] = {}
        groups_data = {g: group_dict[g].get(m, []) for g in group_names}
        for g, scores in groups_data.items():
            result[m][g] = compute_descriptive(scores)
        pairwise = {}
        all_p = []
        for ga, gb in itertools.combinations(group_names, 2):
            u, pu = mann_whitney_u(groups_data[ga], groups_data[gb])
            d = cohens_d(groups_data[ga], groups_data[gb])
            key = f"{ga}_vs_{gb}"
            pairwise[key] = {
                "mann_whitney_U": u, "p_value": pu,
                "cohens_d": d, "effect_size": classify_effect(d),
                "significant_p05": (pu < 0.05) if pu is not None else None,
            }
            if pu is not None:
                all_p.append(pu)
        kw_gs = [groups_data[g] for g in group_names if groups_data[g]]
        h, kw_p = kruskal_wallis(*kw_gs) if len(kw_gs) >= 2 else (None, None)
        result[m]["statistical_tests"] = {
            "kruskal_wallis_H": h,
            "kruskal_wallis_p_value": kw_p,
            "kruskal_wallis_significant_p05": (kw_p < 0.05) if kw_p is not None else None,
            "pairwise_mann_whitney_u": pairwise,
            "bonferroni_correction": bonferroni_correction(all_p),
        }
    return result


def main():
    t0 = time.time()

    ollama_client = OllamaClient(host=OLLAMA_HOST)

    print("[INFO] Memeriksa koneksi Ollama...")

    # models_info = ollama_client.list()
    # print(models_info)


    
    try:
        models_info = ollama_client.list()
        available_models = [m["model"] for m in models_info.get("models", [])]
        print(f"[INFO] Model tersedia di Ollama: {available_models}")
    except Exception as e:
        print(f"[ERROR] Tidak dapat terhubung ke Ollama: {e}")
        print("Pastikan Ollama berjalan dengan: ollama serve")
        sys.exit(1)

    models_to_run = []
    for mdl in MODELS_TO_EVALUATE:
        if any(mdl in am for am in available_models):
            models_to_run.append(mdl)
        else:
            print(f"[WARNING] Model '{mdl}' tidak ditemukan di Ollama. Skip.")
            print(f"          Install dengan: ollama pull {mdl}")

    if not models_to_run:
        print("[ERROR] Tidak ada model yang tersedia. Jalankan minimal: ollama pull llama3")
        sys.exit(1)

    print(f"[INFO] Model yang akan dievaluasi: {models_to_run}")

    all_test_cases_data = []
    for model in models_to_run:
        print(f"\n[INFO] Generating test cases untuk model: {model}")
        tcs = build_test_cases(ollama_client, model)
        all_test_cases_data.extend(tcs)
        print(f"[INFO] Total test cases untuk {model}: {len(tcs)}")

    print(f"\n[INFO] Total seluruh test cases: {len(all_test_cases_data)}")
    print(f"[INFO] Mulai evaluasi komprehensif...\n")

    all_results, raw_by_metric, by_category, by_difficulty, by_model = evaluate_all(
        all_test_cases_data, ollama_client
    )

    print("\n[INFO] Menghitung statistik agregat...")
    aggregate_stats = build_aggregate_statistics(raw_by_metric)
    correlation_matrix = build_correlation_matrix(raw_by_metric)
    model_comparison, global_bonferroni = build_model_comparison(by_model, raw_by_metric)
    category_analysis = build_group_analysis(by_category, "category")
    difficulty_analysis = build_group_analysis(by_difficulty, "difficulty")

    all_composite = [tc["test_case_summary"]["composite_weighted_score"] for tc in all_results if tc["test_case_summary"]["composite_weighted_score"] is not None]
    all_pass_rates = [tc["test_case_summary"]["pass_rate"] for tc in all_results]
    all_times = [tc["test_case_summary"]["execution_time_seconds"] for tc in all_results]
    total_passed = sum(tc["test_case_summary"]["metrics_passed"] for tc in all_results)
    total_evals = sum(tc["test_case_summary"]["total_metrics"] for tc in all_results)
    total_time = round(time.time() - t0, 4)

    comp_ci = confidence_interval_95(all_composite)
    comp_bci = bootstrap_ci(all_composite)
    comp_desc = compute_descriptive(all_composite)

    raw_matrix = {
        tc["test_case_id"]: {
            m["metric_name"]: m["raw_score"] for m in tc["metrics_detail"]
        }
        for tc in all_results
    }

    output_doc = {
        "experiment_metadata": {
            "experiment_id": "EXP_DYSLEXIA_LLM_EVAL_OLLAMA_V4_Q1",
            "title": "Comprehensive Free Multi-Model Evaluation of LLM-Generated Educational Content for Children with Dyslexia Using Local Ollama Models",
            "description": (
                "A fully open-source, zero-cost systematic multi-model evaluation of Large Language Model outputs "
                "for generating educationally appropriate questions and answers tailored to children with dyslexia. "
                "All models run locally via Ollama (no API cost). "
                "Evaluation employs 14 metrics covering: NLP similarity (ROUGE-1/2/L, BLEU), "
                "readability (Flesch Reading Ease, Flesch-Kincaid Grade, Gunning Fog, SMOG, ARI, Coleman-Liau), "
                "dyslexia-specific compliance scoring, LLM-as-Judge (local), "
                "toxicity detection, bias detection, context relevancy, faithfulness, answer relevancy, "
                "and lexical diversity. "
                "Statistical analysis: descriptive statistics, Shapiro-Wilk normality, "
                "Mann-Whitney U, Kruskal-Wallis H, Cohen's d, Pearson r, Spearman rho, "
                "p-values, Bonferroni correction, 95% CI (t-distribution), Bootstrap CI (2000 resamples)."
            ),
            "research_domain": "Special Education Technology / Natural Language Processing / Computational Linguistics",
            "task_type": "Educational Question-Answer Generation for Dyslexic Learners",
            "target_population": "Children with Dyslexia, Elementary School Level, Grade 1-3, Indonesia",
            "evaluation_infrastructure": "Ollama (Local, Free, Open-Source)",
            "models_evaluated": models_to_run,
            "judge_model": JUDGE_MODEL,
            "generator_system_prompt": SYSTEM_PROMPT_DYSLEXIA,
            "timestamp_start_utc": EXPERIMENT_START_UTC,
            "timestamp_end_utc": datetime.now(timezone.utc).isoformat(),
            "total_execution_time_seconds": total_time,
            "python_version": sys.version,
            "platform": platform.platform(),
            "total_test_cases": len(all_results),
            "total_metrics_per_case": len(METRIC_NAMES),
            "total_evaluations_performed": total_evals,
            "cost_usd": 0.0,
            "reproducibility_note": "All models are local. Set same Ollama model versions for full reproducibility.",
        },
        "evaluation_configuration": {
            "metrics_applied": [
                {
                    "metric_name": m,
                    "weight": METRIC_WEIGHTS.get(m, 1.0),
                    "is_inverted": m in INVERTED_METRICS,
                    "threshold": 0.1 if m in {"toxicity","bias"} else 0.5,
                    "computation_method": {
                        "answer_relevancy": "ROUGE-1 + ROUGE-L + BLEU-1 weighted combination vs input prompt",
                        "context_relevancy": "Average ROUGE-1 F1 of output vs each context sentence, scaled",
                        "faithfulness": "Token overlap ratio of output vs context vocabulary, scaled",
                        "toxicity": "Keyword matching against Indonesian toxic word lexicon",
                        "bias": "Phrase matching against Indonesian bias phrase lexicon",
                        "dyslexia_compliance": "Weighted rubric: sentence length (30%), Flesch RE (25%), explicit answer (20%), number range (15%), forbidden words (10%)",
                        "llm_judge_score": "LLM-as-Judge scoring via local Ollama model on 0-1 scale",
                        "rouge1_f1": "ROUGE-1 F1 (unigram overlap) vs expected output",
                        "rouge2_f1": "ROUGE-2 F1 (bigram overlap) vs expected output",
                        "rougeL_f1": "ROUGE-L F1 (LCS-based) vs expected output",
                        "bleu_avg": "Geometric mean of BLEU-1 to BLEU-4 with brevity penalty vs expected output",
                        "flesch_reading_ease": "206.835 - 1.015*(words/sentences) - 84.6*(syllables/words)",
                        "avg_sentence_length_words": "Total words / total sentences (lower = better for dyslexia)",
                        "type_token_ratio": "Unique tokens / total tokens (lexical diversity)",
                    }.get(m, "custom"),
                }
                for m in METRIC_NAMES
            ],
            "composite_score_formula": "Weighted average of normalized metric scores; inverted metrics use (1-score) before normalization",
            "statistical_methods": [
                "Descriptive statistics: mean, std, median, min, max, range, Q1, Q3, IQR, CV, skewness, excess kurtosis",
                "Normality test: Shapiro-Wilk (approximation; W statistic + p-value)",
                "Non-parametric pairwise test: Mann-Whitney U + z-approximation p-value",
                "Non-parametric multi-group test: Kruskal-Wallis H + chi-square approximation p-value",
                "Effect size: Cohen's d with pooled standard deviation",
                "Parametric correlation: Pearson r + t-test p-value",
                "Non-parametric correlation: Spearman rho + t-approximation p-value",
                "Multiple comparisons correction: Bonferroni (family-wise error rate control)",
                "Confidence interval: 95% CI using t-distribution (exact t-critical values for n<30)",
                "Robust CI: Bootstrap 95% CI using 2000 resamples, percentile method, seed=42",
            ],
            "dyslexia_compliance_rubric": {
                "avg_sentence_length_le6_words": "30 points",
                "flesch_reading_ease_ge80": "25 points",
                "has_explicit_answer": "20 points",
                "numbers_in_range_1_20": "15 points",
                "no_forbidden_complex_words": "10 points",
            },
        },
        "test_cases": all_results,
        "aggregate_statistics_per_metric": aggregate_stats,
        "metric_correlation_matrix": correlation_matrix,
        "model_comparison_analysis": model_comparison,
        "global_bonferroni_correction_all_pairwise": global_bonferroni,
        "performance_by_category": category_analysis,
        "performance_by_difficulty_level": difficulty_analysis,
        "overall_experiment_summary": {
            "total_test_cases": len(all_results),
            "models_evaluated": models_to_run,
            "total_evaluations_performed": total_evals,
            "total_metrics_passed": total_passed,
            "total_metrics_failed": total_evals - total_passed,
            "overall_pass_rate": round(total_passed/total_evals, 6) if total_evals > 0 else 0,
            "composite_score_statistics": comp_desc,
            "composite_score_ci_95_t": {"lower": comp_ci[0], "upper": comp_ci[1]},
            "composite_score_bootstrap_ci_95": {"lower": comp_bci[0], "upper": comp_bci[1]},
            "pass_rate_statistics": compute_descriptive(all_pass_rates),
            "execution_time_statistics": {
                "total_seconds": total_time,
                "mean_per_case_seconds": round(safe_mean(all_times), 4) if all_times else None,
                "min_seconds": round(min(all_times), 4) if all_times else None,
                "max_seconds": round(max(all_times), 4) if all_times else None,
            },
        },
        "raw_scores_matrix": raw_matrix,
        "readability_summary_per_model": {
            mdl: {
                "flesch_reading_ease": compute_descriptive([
                    tc["readability_analysis"]["flesch_reading_ease"]
                    for tc in all_results if tc["model"] == mdl
                ]),
                "avg_sentence_length_words": compute_descriptive([
                    tc["readability_analysis"]["avg_sentence_length_words"]
                    for tc in all_results if tc["model"] == mdl
                ]),
                "flesch_kincaid_grade": compute_descriptive([
                    tc["readability_analysis"]["flesch_kincaid_grade"]
                    for tc in all_results if tc["model"] == mdl
                ]),
                "gunning_fog_index": compute_descriptive([
                    tc["readability_analysis"]["gunning_fog_index"]
                    for tc in all_results if tc["model"] == mdl
                ]),
                "smog_index": compute_descriptive([
                    tc["readability_analysis"]["smog_index"]
                    for tc in all_results if tc["model"] == mdl
                ]),
                "automated_readability_index": compute_descriptive([
                    tc["readability_analysis"]["automated_readability_index"]
                    for tc in all_results if tc["model"] == mdl
                ]),
                "coleman_liau_index": compute_descriptive([
                    tc["readability_analysis"]["coleman_liau_index"]
                    for tc in all_results if tc["model"] == mdl
                ]),
                "type_token_ratio": compute_descriptive([
                    tc["readability_analysis"]["type_token_ratio"]
                    for tc in all_results if tc["model"] == mdl
                ]),
                "pct_sentences_exceeding_8_words": compute_descriptive([
                    tc["readability_analysis"]["sentences_exceeding_8_words_pct"]
                    for tc in all_results if tc["model"] == mdl
                ]),
                "dyslexia_compliance_score": compute_descriptive([
                    tc["test_case_summary"]["dyslexia_compliance_score"]
                    for tc in all_results if tc["model"] == mdl
                ]),
            }
            for mdl in models_to_run
        },
    }

    output_path = "evaluasi_llm_disleksia_scopus_q1_ollama_final.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output_doc, f, indent=2, ensure_ascii=False)

    print(f"\n{'='*65}")
    print(f"  Evaluasi selesai!")
    print(f"  Output: {output_path}")
    print(f"  Total test cases  : {len(all_results)}")
    print(f"  Model dievaluasi  : {models_to_run}")
    print(f"  Total evaluasi    : {total_evals}")
    print(f"  Overall pass rate : {round(total_passed/total_evals*100,2) if total_evals else 0}%")
    print(f"  Composite mean    : {round(safe_mean(all_composite),4) if all_composite else 'N/A'}")
    print(f"  Total waktu       : {total_time} detik")
    print(f"  Biaya API         : $0.00 (Ollama lokal)")
    print(f"{'='*65}")


if __name__ == "__main__":
    main()

[INFO] Memeriksa koneksi Ollama...
[INFO] Model tersedia di Ollama: ['gemma:latest', 'llama3:latest', 'mistral:latest']
[INFO] Model yang akan dievaluasi: ['llama3']

[INFO] Generating test cases untuk model: llama3
[INFO] Total test cases untuk llama3: 62

[INFO] Total seluruh test cases: 62
[INFO] Mulai evaluasi komprehensif...

  [1/62] Evaluating: TC_MAT_PENJUMLAHAN_LLAMA3_001
  [2/62] Evaluating: TC_MAT_PENJUMLAHAN_LLAMA3_002
  [3/62] Evaluating: TC_MAT_PENJUMLAHAN_LLAMA3_003
  [4/62] Evaluating: TC_MAT_PENJUMLAHAN_LLAMA3_004
  [5/62] Evaluating: TC_MAT_PENJUMLAHAN_LLAMA3_005
  [6/62] Evaluating: TC_MAT_PENJUMLAHAN_LLAMA3_006
  [7/62] Evaluating: TC_MAT_PENJUMLAHAN_LLAMA3_007
  [8/62] Evaluating: TC_MAT_PENJUMLAHAN_LLAMA3_008
  [9/62] Evaluating: TC_MAT_PENGURANGAN_LLAMA3_001
  [10/62] Evaluating: TC_MAT_PENGURANGAN_LLAMA3_002
  [11/62] Evaluating: TC_MAT_PENGURANGAN_LLAMA3_003
  [12/62] Evaluating: TC_MAT_PENGURANGAN_LLAMA3_004
  [13/62] Evaluating: TC_MAT_PENGURANGAN_LLAMA3_005
